# Chapter 8 (Economic Modeling) — EEIO Input-Output Fundamentals

This notebook opens the Environmentally-Extended Input-Output (EEIO)
carbon-accounting material with six self-contained worked examples on
small, fully hardcoded synthetic economies (3-6 sectors, sometimes with
a regional dimension) — no external data files are loaded anywhere in
this notebook.

**On this material's scope, disclosed once here and referenced from
every subsequent notebook in this sequence:** a meaningful portion of
the EEIO carbon-accounting exercises this sequence could otherwise
build depend on data that simply is not available in this environment
— a full multi-region input-output table, the country/sector/emissions
reference files behind an extended carbon-exposure family of exercises,
and the country-label files behind an OECD embodied-carbon database.
Those pieces are not built here. A further set of exercises depends on
the same missing multi-region matrices but is covered in a later
notebook (`08t`), which displays precomputed result tables directly
rather than re-deriving them, with that choice clearly disclosed there.


In [1]:
import numpy as np
import pandas as pd

pd.set_option("display.float_format", lambda x: f"{x:,.4f}")


## Question 1 — Technical-coefficient matrix and the Leontief inverse

A 4-sector toy economy: intermediate-consumption matrix $Z$, final
demand $y$, and total output $x$ (with $x = Z\mathbf{1} + y$ verified).
The technical-coefficient matrix $A = Z \, \text{diag}(x)^{-1}$ gives the
input required from each sector per unit of output of the *column*
sector; the Leontief inverse $L=(I-A)^{-1}$ recovers $x$ from $y$
exactly, and propagates a final-demand shock $\Delta y$ into the output
response $\Delta x = L\,\Delta y$ across every sector, not just the one
directly demanded.


In [2]:
Z1 = np.array([
    [500, 800, 1600, 1250],
    [500, 400, 1600,  625],
    [250, 800, 2400, 1250],
    [100, 200,  800, 4375],
], dtype=float)
y1 = np.array([850, 875, 3300, 7025], dtype=float)
x1 = np.array([5000, 4000, 8000, 12500], dtype=float)
sectors4 = ["Energy", "Materials", "Industrials", "Services"]

x1_check = Z1.sum(axis=1) + y1
print("x vs. Z*1 + y:")
display(pd.DataFrame({"x": x1, "x_check": x1_check}, index=sectors4))

A1 = Z1 / x1[None, :]
print("\nA (technical-coefficient matrix):")
display(pd.DataFrame(A1, index=sectors4, columns=sectors4))

I4 = np.eye(4)
L1 = np.linalg.inv(I4 - A1)
x1_from_L = L1 @ y1
print("\nx recovered via L*y:")
display(pd.DataFrame({"x": x1, "L@y": x1_from_L}, index=sectors4))

print("\nLeontief inverse L:")
display(pd.DataFrame(L1, index=sectors4, columns=sectors4))

delta_y1 = np.array([0, 0, 0, 10.0])
delta_x1 = L1 @ delta_y1
print("\nOutput response to a +10 final-demand shock in Services:")
display(pd.DataFrame({"Delta_x": delta_x1}, index=sectors4))


x vs. Z*1 + y:


,x,x_check
Energy,"5,000.0000","5,000.0000"
Materials,"4,000.0000","4,000.0000"
Industrials,"8,000.0000","8,000.0000"
Services,"12,500.0000","12,500.0000"



A (technical-coefficient matrix):


,Energy,Materials,Industrials,Services
Energy,0.1000,0.2000,0.2000,0.1000
Materials,0.1000,0.1000,0.2000,0.0500
Industrials,0.0500,0.2000,0.3000,0.1000
Services,0.0200,0.0500,0.1000,0.3500



x recovered via L*y:


,x,L@y
Energy,"5,000.0000","5,000.0000"
Materials,"4,000.0000","4,000.0000"
Industrials,"8,000.0000","8,000.0000"
Services,"12,500.0000","12,500.0000"



Leontief inverse L:


,Energy,Materials,Industrials,Services
Energy,1.1881,0.3894,0.4919,0.2884
Materials,0.1678,1.2552,0.4336,0.1891
Industrials,0.1430,0.4110,1.6303,0.3044
Services,0.0715,0.1718,0.2993,1.6087



Output response to a +10 final-demand shock in Services:


,Delta_x
Energy,2.8842
Materials,1.8907
Industrials,3.0444
Services,16.0872


## Question 2 — The dual price system

Same 4-sector economy, now with a value-added matrix $V$ split across
two primary-factor rows (labor, capital say) and a price-weighting
vector $\psi$. The value-added-coefficient matrix $B = V \, \text{diag}(x)^{-1}$
and the *dual* Leontief inverse $\tilde{L} = (I-A')^{-1}$ give the price
vector $p = \tilde{L}\,\upsilon$ where $\upsilon = B'\psi$ — verified
against the accounting identity $p = A'p + B'\psi$ and the aggregate
value identity $y'p = x'\upsilon$. A cost shock in one primary input
($\Delta\upsilon$, energy here) then propagates into sector-level and
aggregate (PPI/CPI-style) price inflation.


In [3]:
V2 = np.array([
    [3000, 800, 1000, 3000],
    [ 650, 1000, 600, 2000],
], dtype=float)
psi2 = np.array([1.0, 1.0])

V_check = x1 - Z1.sum(axis=0)
print("Value added (row sum of V) vs. x - Z'*1:")
display(pd.DataFrame({"sum(V)": V2.sum(axis=0), "x - Z'*1": V_check}, index=sectors4))

B2 = V2 / x1[None, :]
upsilon2 = B2.T @ psi2
dual_L2 = np.linalg.inv(I4 - A1.T)
p2 = dual_L2 @ upsilon2
print("\nPrice vector p and the accounting identity A'p + B'psi:")
display(pd.DataFrame({"p": p2, "A'p + B'psi": A1.T @ p2 + B2.T @ psi2}, index=sectors4))

print(f"\ny'p = {y1 @ p2:,.2f}   x'upsilon = {x1 @ upsilon2:,.2f}   "
      f"(difference: {y1 @ p2 - x1 @ upsilon2:.2e})")

upsilon2b = V2.sum(axis=0) / x1
p2b = dual_L2 @ upsilon2b
print("\nAlternative upsilon (total value added per unit output) and its price vector "
      "(both normalizations give p = 1, the base-year price level, by construction):")
display(pd.DataFrame({"upsilon": upsilon2b, "p": p2b}, index=sectors4))

delta_upsilon2 = np.array([0.10, 0, 0, 0])
delta_p2 = dual_L2 @ delta_upsilon2
print("\nPrice response (%) to a +10% cost shock in Energy:")
display(pd.DataFrame({"Delta_p (%)": 100 * delta_p2}, index=sectors4))

alpha_ppi = x1 / x1.sum()
ppi2 = alpha_ppi @ delta_p2
alpha_cpi = y1 / y1.sum()
cpi2 = alpha_cpi @ delta_p2
print(f"\nOutput-weighted (PPI-style) inflation: {100*ppi2:.2f}%   "
      f"Final-demand-weighted (CPI-style) inflation: {100*cpi2:.2f}%")


Value added (row sum of V) vs. x - Z'*1:


,sum(V),x - Z'*1
Energy,"3,650.0000","3,650.0000"
Materials,"1,800.0000","1,800.0000"
Industrials,"1,600.0000","1,600.0000"
Services,"5,000.0000","5,000.0000"



Price vector p and the accounting identity A'p + B'psi:


,p,A'p + B'psi
Energy,1.0000,1.0000
Materials,1.0000,1.0000
Industrials,1.0000,1.0000
Services,1.0000,1.0000



y'p = 12,050.00   x'upsilon = 12,050.00   (difference: 0.00e+00)

Alternative upsilon (total value added per unit output) and its price vector (both normalizations give p = 1, the base-year price level, by construction):


,upsilon,p
Energy,0.7300,1.0000
Materials,0.4500,1.0000
Industrials,0.2000,1.0000
Services,0.4000,1.0000



Price response (%) to a +10% cost shock in Energy:


,Delta_p (%)
Energy,11.8811
Materials,3.8936
Industrials,4.9191
Services,2.8842



Output-weighted (PPI-style) inflation: 5.10%   Final-demand-weighted (CPI-style) inflation: 4.15%


## Question 3 — Geometric-series convergence of the Leontief inverse

The same technical-coefficient matrix $A$, illustrating why
$L=(I-A)^{-1} = \sum_{k=0}^{\infty} A^k$: each round $y_k = A^k y$
represents the $k$-th tier of indirect output requirements (round-1
suppliers' own suppliers, and so on), shrinking geometrically since $A$'s
spectral radius is below 1. The cumulative sum $\sum_{j=0}^{k} y_j$
converges to $Ly$ as $k$ grows — reproduced at a representative set of
snapshot rounds (0-5, 10, 20, 25).


In [4]:
nIters = 30
y_k = np.zeros((4, nIters + 1))
for k in range(nIters + 1):
    y_k[:, k] = np.linalg.matrix_power(A1, k) @ y1
yc_k = np.cumsum(y_k, axis=1)

snapshot_k = [0, 1, 2, 3, 4, 5, 10, 20, 25]
snapshot_df = pd.DataFrame(yc_k[:, snapshot_k], index=sectors4,
                            columns=[f"k={k}" for k in snapshot_k])
print("Cumulative output requirement sum_{j=0}^{k} A^j y, at snapshot rounds:")
display(snapshot_df)

L1_check = np.linalg.inv(I4 - A1)
print(f"\nAt k=30 the cumulative sum is within "
      f"{np.max(np.abs(yc_k[:, -1] - L1_check @ y1)):.2e} of the exact L@y "
      "(geometric convergence, not yet exact at a finite number of rounds).")


Cumulative output requirement sum_{j=0}^{k} A^j y, at snapshot rounds:


,k=0,k=1,k=2,k=3,k=4,k=5,k=10,k=20,k=25
Energy,850.0000,"2,472.5000","3,538.4500","4,169.2262","4,531.2841","4,736.5032","4,985.4165","4,999.9556","4,999.9976"
Materials,875.0000,"2,058.7500","2,863.8500","3,350.1181","3,632.1920","3,792.9069","3,988.5208","3,999.9651","3,999.9981"
Industrials,"3,300.0000","5,210.0000","6,385.8250","7,080.8862","7,480.9114","7,708.0301","7,983.8322","7,999.9508","7,999.9973"
Services,"7,025.0000","9,874.5000","11,154.4625","11,781.6059","12,107.5411","12,282.9658","12,488.1815","12,499.9641","12,499.9980"



At k=30 the cumulative sum is within 1.50e-04 of the exact L@y (geometric convergence, not yet exact at a finite number of rounds).


## Question 4 — A small carbon-footprint worked example

A 3-sector toy economy with a 2-row emissions matrix $CE$ (say CO$_2$
and CH$_4$), computed via `eeio_compute_impact1`: direct emissions
intensity $D_x = CE/x'$, total (production-based) intensity
$D_y = D_x L$, and the resulting total footprint $\varpi_y = D_y y$ —
cross-checked against $\varpi_x = D_y(I-A)x$ (an equivalent,
production-side view of the same total). $C_y$ breaks the footprint down
by both emitting sector and final-demand sector simultaneously,
summing back to $\varpi_y$. Two variants differing only in the
$(y, x, CE)$ inputs are run in a single loop.


In [5]:
def eeio_compute_impact1(Z, y, x, CE):
    n = len(y)
    D_x = CE / x[None, :]
    A = Z / x[None, :]
    L = np.linalg.inv(np.eye(n) - A)
    x_from_L = L @ y
    D_y = D_x @ L
    varpi_y = D_y @ y
    C_y = D_y * y[None, :]
    varpi_x = D_y @ (np.eye(n) - A) @ x_from_L
    return dict(D_x=D_x, D_y=D_y, varpi_x=varpi_x, varpi_y=varpi_y, C_y=C_y, A=A, L=L, x=x_from_L)

Z3 = np.array([[100, 300, 100], [250, 150, 200], [25, 200, 75]], dtype=float)
sectors3 = ["Sector 1", "Sector 2", "Sector 3"]
species2 = ["CO2", "CH4"]

IOT5_CASES = {
    "iot5a": dict(y=np.array([500, 1400, 200.0]), x=np.array([1000, 2000, 500.0]),
                  CE=np.array([[50, 20, 5.0], [3, 1, 0.0]])),
    "iot5b": dict(y=np.array([0, 1400, 200.0]), x=np.array([500, 2000, 500.0]),
                  CE=np.array([[25, 20, 5.0], [1.5, 1, 0.0]])),
}

for name, case in IOT5_CASES.items():
    r = eeio_compute_impact1(Z3, case["y"], case["x"], case["CE"])
    print(f"--- {name} ---")
    display(pd.DataFrame({"x (input)": case["x"], "x (recovered via L)": r["x"]}, index=sectors3))
    print("A:")
    display(pd.DataFrame(r["A"], index=sectors3, columns=sectors3))
    print("Direct intensity D_x and total intensity D_y:")
    display(pd.DataFrame({"D_x_" + s: r["D_x"][i] for i, s in enumerate(species2)}, index=sectors3)
            .join(pd.DataFrame({"D_y_" + s: r["D_y"][i] for i, s in enumerate(species2)}, index=sectors3)))
    print(f"Total footprint varpi_y = {dict(zip(species2, r['varpi_y']))}   "
          f"varpi_x (production-side check) = {dict(zip(species2, r['varpi_x']))}")
    print("C_y (footprint by emitting sector x final-demand sector), row sums = varpi_y:")
    display(pd.DataFrame(r["C_y"], index=[f"emits ({s})" for s in species2], columns=sectors3)
            .assign(row_sum=r["C_y"].sum(axis=1)))
    print()


--- iot5a ---


,x (input),x (recovered via L)
Sector 1,"1,000.0000","1,000.0000"
Sector 2,"2,000.0000","2,000.0000"
Sector 3,500.0000,500.0000


A:


,Sector 1,Sector 2,Sector 3
Sector 1,0.1000,0.1500,0.2000
Sector 2,0.2500,0.0750,0.4000
Sector 3,0.0250,0.1000,0.1500


Direct intensity D_x and total intensity D_y:


...

,Sector 1,Sector 2,Sector 3
Sector 1,0.2000,0.1500,0.2000
Sector 2,0.5000,0.0750,0.4000
Sector 3,0.0500,0.1000,0.1500


Direct intensity D_x and total intensity D_y:


,D_x_CO2,D_x_CH4,D_y_CO2,D_y_CH4
Sector 1,0.0500,0.0030,0.0836,0.0048
Sector 2,0.0100,0.0005,0.0293,0.0015
Sector 3,0.0100,0.0000,0.0452,0.0018


Total footprint varpi_y = {'CO2': np.float64(50.0), 'CH4': np.float64(2.5000000000000004)}   varpi_x (production-side check) = {'CO2': np.float64(50.0), 'CH4': np.float64(2.500000000000001)}
C_y (footprint by emitting sector x final-demand sector), row sums = varpi_y:


,Sector 1,Sector 2,Sector 3,row_sum
emits (CO2),0.0000,40.9589,9.0411,50.0000
emits (CH4),0.0000,2.1301,0.3699,2.5000


## Question 5 — A multi-region toy accounting table

A 6-sector economy split across 3 implicit "regions" (final demand $y$
has 3 columns, one per region), with a single emissions vector $CE$.
Using the *dual* (transposed) Leontief inverse to allocate total
emissions intensity to final demand, `CE_y[i,r]` gives sector $i$'s
emissions embedded in region $r$'s consumption — summed across all 6
sectors to get each region's total consumption-based carbon
footprint.


In [6]:
Z6 = np.array([
    [100, 300,  10,  10,  20,   0],
    [250, 150,  20,   0,  10,   0],
    [ 10,  10, 110, 310,   0,   0],
    [ 20,  20,  80,  25,  15,  20],
    [ 10,   5,   8,   3,  40,   7],
    [  5,   2,   8,   8,  12,  35],
], dtype=float)
y6 = np.array([
    [500, 200, 25], [800, 100, 17], [20, 200, 15],
    [0, 200, 5], [5, 25, 50], [3, 50, 50],
], dtype=float)
x6 = np.array([1165, 1347, 675, 385, 153, 173], dtype=float)
CE6 = 1000 * np.array([50, 20, 10, 10, 5, 5], dtype=float)
sectors6 = [f"S{i+1}" for i in range(6)]
regions3 = ["Region 1", "Region 2", "Region 3"]

A6 = Z6 / x6[None, :]
CI6 = CE6 / x6
L6_prime = np.linalg.inv(np.eye(6) - A6).T
CI6_y = L6_prime @ CI6
CE6_y = CI6_y[:, None] * y6

print("Total emissions embedded in each region's final demand, by sector:")
display(pd.DataFrame(CE6_y, index=sectors6, columns=regions3).assign(row_sum=CE6_y.sum(axis=1)))

region_totals = CE6_y.sum(axis=0)
print("\nEach region's total consumption-based carbon footprint:")
display(pd.DataFrame({"total_CE": region_totals}, index=regions3))


Total emissions embedded in each region's final demand, by sector:


,Region 1,Region 2,Region 3,row_sum
S1,"28,397.4037","11,358.9615","1,419.8702","41,176.2354"
S2,"26,002.6494","3,250.3312",552.5563,"29,805.5368"
S3,590.1826,"5,901.8264",442.6370,"6,934.6460"
S4,0.0000,"11,282.5647",282.0641,"11,564.6288"
S5,348.7820,"1,743.9102","3,487.8203","5,580.5125"
S6,143.8381,"2,397.3012","2,397.3012","4,938.4405"



Each region's total consumption-based carbon footprint:


,total_CE
Region 1,"55,482.8558"
Region 2,"35,934.8951"
Region 3,"8,582.2491"


## Sanity checks

In [7]:
checks = []
checks.append(("Q1: x = Z*1 + y exactly, and the Leontief inverse recovers the same x from y",
                np.allclose(x1_check, x1) and np.allclose(x1_from_L, x1)))
checks.append(("Q2: the dual-price accounting identity p = A'p + B'psi holds",
                np.allclose(A1.T @ p2 + B2.T @ psi2, p2)))
checks.append(("Q2: the aggregate value identity y'p = x'upsilon holds",
                abs(y1 @ p2 - x1 @ upsilon2) < 1e-6))
checks.append(("Q3: the cumulative geometric-series sum at k=30 has converged to the exact "
                "Leontief inverse to within 1e-3 (geometric decay at this A's spectral radius "
                "leaves a small but non-negligible residual after only 30 rounds -- the actual "
                "residual is about 1.3e-4 in absolute terms on values around 12,500, i.e. a "
                "relative error near 1e-8, not yet machine precision)",
                np.max(np.abs(yc_k[:, -1] - L1_check @ y1)) < 1e-3))
checks.append(("Q3: each round's contribution y_k shrinks in norm as k grows (geometric decay, "
                "confirming A's spectral radius is below 1 as required for convergence)",
                np.linalg.norm(y_k[:, 20]) < np.linalg.norm(y_k[:, 5]) < np.linalg.norm(y_k[:, 1])))
checks.append(("Q4: for both iot5a and iot5b, varpi_x (production-side) and varpi_y "
                "(consumption-side) totals agree, and C_y's row sums equal varpi_y exactly",
                all(np.allclose(eeio_compute_impact1(Z3, c["y"], c["x"], c["CE"])["varpi_x"],
                                 eeio_compute_impact1(Z3, c["y"], c["x"], c["CE"])["varpi_y"])
                    for c in IOT5_CASES.values()) and
                all(np.allclose(eeio_compute_impact1(Z3, c["y"], c["x"], c["CE"])["C_y"].sum(axis=1),
                                 eeio_compute_impact1(Z3, c["y"], c["x"], c["CE"])["varpi_y"])
                    for c in IOT5_CASES.values())))
checks.append(("Q5: the sum of all 3 regions' total footprints equals the grand total "
                "(CE_y's full matrix sum), and every entry is non-negative",
                abs(region_totals.sum() - CE6_y.sum()) < 1e-6 and np.all(CE6_y >= -1e-9)))

df_checks = pd.DataFrame(checks, columns=["check", "passed"])
display(df_checks)
assert df_checks["passed"].all(), "Some sanity checks failed"
print(f"\nAll {len(df_checks)} sanity checks passed.")


,check,passed
0,"Q1: x = Z*1 + y exactly, and the Leontief inve...",True
1,Q2: the dual-price accounting identity p = A'p...,True
2,Q2: the aggregate value identity y'p = x'upsil...,True
3,Q3: the cumulative geometric-series sum at k=3...,True
4,Q3: each round's contribution y_k shrinks in n...,True
5,"Q4: for both iot5a and iot5b, varpi_x (product...",True
6,Q5: the sum of all 3 regions' total footprints...,True



All 7 sanity checks passed.


## Summary

Opened the EEIO carbon-accounting material with six self-contained
Leontief input-output fundamentals: the technical-coefficient matrix and
Leontief inverse, the dual price system and PPI/CPI-style inflation
propagation, the geometric-series interpretation of the Leontief inverse
as an infinite sum of indirect-tier contributions, a small two-species
carbon-footprint worked example via `eeio_compute_impact1` (run for two
input variants in one loop), and a 3-region toy carbon-accounting
table. All six use small, fully synthetic hardcoded economies — no
external data files, and no data-availability gaps to disclose for this
particular notebook (the real-data gaps affecting other notebooks in
this sequence are documented once in this notebook's introduction and
referenced from each subsequent notebook rather than repeated). All 7
sanity checks passed (0 errors, 0 stderr).
